In [0]:
dbutils.widgets.removeAll()

In [0]:
import re
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
# 1. Detección por ASUNTO (regex, orden importa: primer match gana)
PRODUCTOS_POR_ASUNTO = [
    ("PROMERICA",   [r"promerica", r"bp\b"]),
    ("CREDOMATIC",    [r"\bbac\b", r"credomatic"]),
    ("BI 90",  [r"banco industrial", r"\bbi\s*9[0-9]\b", r"\bbi\s*6[0-9]\b",
                           r"\bbi\b", r"bi90", r"bi60", r"biwoff", r"bi woff"]),
    ("TARJETA CJ",   [r"g&t", r"g\+t", r"continental"])
]

# 2. Fallback por REMITENTE (exacto, sin distinción may/min)
PRODUCTOS_POR_REMITENTE = {
    "ic.fajardo@ocacall.com":   "CREDOMATIC",
    "j.suarez@ocacall.com":     "BI WOFF",
    "ed.morales@ocacall.com":   "PROMERICA",
    "a.morales@ocacall.com":    "PROMERICA",
    "j.hernandez@ocacall.com":  "TARJETA CJ",
    "h.gonzalez@ocacall.com":   "BAC ADMIN",
    "ld.garcia@ocacall.com":    "BI 90",
    "a.alvarez@ocacall.com":    "BI WOFF",
    "j.tanchez@ocacall.com":    "BI WOFF",
    "l.rivera@ocacall.com":     "TARJETA CJ",
}

**UDF**

In [0]:
def detectar_producto(asunto, remitente):
    if asunto is not None:
        a = str(asunto).lower()

        for nombre, patrones in PRODUCTOS_POR_ASUNTO:
            for p in patrones:
                if re.search(p, a):
                    return nombre

    if remitente is not None:
        r = str(remitente).strip().lower()

        if r in PRODUCTOS_POR_REMITENTE:
            return PRODUCTOS_POR_REMITENTE[r]

    return "Sin clasificar"

detectar_producto_udf = udf(detectar_producto, StringType())

In [0]:
df_catalogo = spark.table(f"{catalogo}.{esquema_sink}.catalogo_transformed")
df_email = spark.table(f"{catalogo}.{esquema_source}.emails")

In [0]:
df_catalogo_unico = (
    df_catalogo
    .groupBy("producto")
    .agg(first("cliente").alias("cliente"))
)

In [0]:
df_email = df_email.dropna(how="all")\
                        .filter(col("senderaddress").isNotNull())\
                        .filter(col("recipientaddress").isNotNull())\
                        .filter(col("subject").isNotNull())

In [0]:
df_email = df_email.withColumn("anio", year(col("received")))

df_email = df_email.withColumn("mes", month(col("received")))
df_email = df_email.withColumn("producto", detectar_producto_udf(col("subject"), col("senderaddress")))


In [0]:
df_email_catalogo = (
    df_email
    .join(df_catalogo_unico, on="producto", how="left")
    .withColumn(
        "cliente",
        coalesce("cliente", lit("Sin clasificar"))
    )
)

In [0]:
df_email_final = df_email_catalogo.select(
    col("anio"),
    col("mes"),
    col("cliente"),
    col("producto"),
    col("received"),
    col("senderaddress"),
    col("recipientaddress"),
    col("subject"),
    col("status"),
    col("messagetraceid"),
    col("ingestion_date")
)


In [0]:
df_email_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.emails_transformed")